<a href="https://colab.research.google.com/github/trisha-agni/sentence_transformers/blob/main/SentenceTransformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print(model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


Basic encoding

In [2]:
sentences = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",
    "The stock market crashed yesterday.",
]

embeddings = model.encode(sentences)
print(embeddings.shape)  # (3, 384) for MiniLM
print(embeddings[0][:10])  # peek at raw values

(3, 384)
[ 0.13023719 -0.01577281 -0.03671669  0.05798644 -0.05979173  0.03305371
  0.03012399  0.02892722 -0.01860521  0.05529655]


Semantic similarity

In [3]:
from sentence_transformers import util

emb1 = model.encode("The cat sat on the mat.")
emb2 = model.encode("A feline rested on the rug.")
emb3 = model.encode("The stock market crashed yesterday.")

print("Cat vs feline:", util.cos_sim(emb1, emb2).item())
print("Cat vs stock market:", util.cos_sim(emb1, emb3).item())

Cat vs feline: 0.5559762716293335
Cat vs stock market: 0.11098036915063858


Semantic search over a small corpus

In [4]:
corpus = [
    "Python is a popular programming language.",
    "The weather today is sunny and warm.",
    "Machine learning models require training data.",
    "I enjoy hiking in the mountains.",
    "Neural networks are utilized in deep learning.",
]
corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

query = "What is AI used for?"
query_embedding = model.encode(query, convert_to_tensor=True)

hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)[0]
for hit in hits:
    print(f"{corpus[hit['corpus_id']]} (score: {hit['score']:.4f})")

Neural networks are utilized in deep learning. (score: 0.4604)
Machine learning models require training data. (score: 0.3019)
Python is a popular programming language. (score: 0.2845)


Duplicate Detection

In [5]:
questions = [
    "How do I reset my password?",
    "What's the process for password recovery?",
    "How can I change my email address?",
    "Where do I update my contact email?",
]

embeddings = model.encode(questions)
sim_matrix = util.cos_sim(embeddings, embeddings)
print(sim_matrix)

threshold = 0.6
for i in range(len(questions)):
    for j in range(i+1, len(questions)):
        if sim_matrix[i][j] > threshold:
            print(f"Possible duplicates:\n  '{questions[i]}'\n  '{questions[j]}'\n  score={sim_matrix[i][j]:.3f}\n")

tensor([[1.0000, 0.6330, 0.4316, 0.3156],
        [0.6330, 1.0000, 0.1160, 0.1483],
        [0.4316, 0.1160, 1.0000, 0.6487],
        [0.3156, 0.1483, 0.6487, 1.0000]])
Possible duplicates:
  'How do I reset my password?'
  'What's the process for password recovery?'
  score=0.633

Possible duplicates:
  'How can I change my email address?'
  'Where do I update my contact email?'
  score=0.649



Saving Embeddings to the Disk

In [6]:
import torch

corpus = [
    "Python is a popular programming language.",
    "The weather today is sunny and warm.",
    "Machine learning models require training data.",
    "I enjoy hiking in the mountains.",
    "Neural networks are used in deep learning.",
]

corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

torch.save(corpus_embeddings, "corpus_embeddings.pt")

In [8]:
loaded_embeddings = torch.load("corpus_embeddings.pt")

query_embedding = model.encode("What is AI used for?", convert_to_tensor=True)
hits = util.semantic_search(query_embedding, loaded_embeddings, top_k=3)[0]

for hit in hits:
    print(f"{corpus[hit['corpus_id']]} (score: {hit['score']:.4f})")

Neural networks are used in deep learning. (score: 0.5273)
Machine learning models require training data. (score: 0.3019)
Python is a popular programming language. (score: 0.2845)


The Application of Sentence-Transformers in RAG

In [11]:
# Step 1: encode a knowledge base once and save it
documents = [
    "Shipping typically takes 5-7 business days within the country.",
    "Customers can contact support via email or live chat.",
    "Our refund policy allows returns within 30 days of purchase.",
    "Premium members receive free expedited shipping on all orders.",
]

doc_embeddings = model.encode(documents, convert_to_tensor=True)
torch.save(doc_embeddings, "knowledge_base.pt")

In [12]:
# Step 2: at query time, load the cached embeddings and retrieve relevant context
loaded_embeddings = torch.load("knowledge_base.pt")

user_question = "How long do I have to return something?"
query_embedding = model.encode(user_question, convert_to_tensor=True)

hits = util.semantic_search(query_embedding, loaded_embeddings, top_k=2)[0]
retrieved_context = [documents[hit['corpus_id']] for hit in hits]

print(retrieved_context)

['Our refund policy allows returns within 30 days of purchase.', 'Shipping typically takes 5-7 business days within the country.']


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

prompt = f"""Answer the question using only the context below.

Context:
{chr(10).join(retrieved_context)}

Question: {user_question}
"""

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model_llm.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

30 days
